# 06 — Teste do `PhoneScorer`

Uso mínimo do algoritmo:

1. Carrega as **fontes auxiliares** (`deliverables/` e `outputs/`) e
   `cpf_telefones.parquet` (gerado em `01_preprocessing.ipynb`).
2. Instancia `PhoneScorer` via `from_paths`.
3. Filtra um CPF que aparece em vários sistemas e roda `score` / `top_k`.

A entrada principal é o DataFrame fino

`(cpf, telefone, id_sistema, data_atualizacao)` — com repetições. O algoritmo
consolida internamente: data mais recente por `(cpf, telefone, id_sistema)` e
sistema mais confiável por `(cpf, telefone)`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from phone_scorer import PhoneScorer, ScoreWeights

RANKING_PATH = ROOT / 'deliverables' / 'ranking_confiabilidade.csv'
REGRAS_PATH = ROOT / 'outputs' / 'decision_trees' / 'df_regras_atualidade.csv'
TAXA_READ_PATH = ROOT / 'outputs' / 'processed' / 'taxa_read_telefone.csv'
CPF_TELEFONES_PATH = ROOT / 'outputs' / 'processed' / 'cpf_telefones.parquet'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Carrega dados e instancia o `PhoneScorer`

Fontes auxiliares vão no construtor; a entrada principal (telefones por CPF)
é passada depois em `score(...)`.

In [2]:
df_cpf_telefones = pd.read_parquet(CPF_TELEFONES_PATH)
print('shape:', df_cpf_telefones.shape)
df_cpf_telefones.head()

shape: (1529167, 4)


,cpf,telefone,id_sistema,data_atualizacao
0,5073517428359850284,11583939707640169990,1257277410380486863,2024-10-30
1,4935468162812723950,3856002700049294556,3094574413675758272,2023-09-01
2,14116685989763717186,8067166217402075300,3094574413675758272,2025-10-08
3,1403367098613331454,1478899732613896317,4458959843028638627,2025-02-22
4,5453937523093464096,12340928344900733458,1257277410380486863,2023-06-06


In [3]:
scorer = PhoneScorer.from_paths(
    ranking_path=RANKING_PATH,
    regras_path=REGRAS_PATH,
    taxa_read_path=TAXA_READ_PATH,
    weights=ScoreWeights(w_sistema=0.3, w_atualidade=0.2, w_read=0.5),
)

## 2. Escolhe um CPF de exemplo

Pegamos um CPF com **vários telefones** e com `(cpf, telefone)` em **mais de
um sistema** — exercita as duas regras de consolidação interna.

In [4]:
n_sis = (
    df_cpf_telefones.groupby(['cpf', 'telefone'])['id_sistema']
    .nunique().rename('n_sistemas').reset_index()
)
candidatos = (
    n_sis[n_sis['n_sistemas'] >= 2]
    .groupby('cpf').size().rename('n_telefones_multi').reset_index()
    .sort_values('n_telefones_multi', ascending=False)
)
print('CPFs candidatos (top 5):')
display(candidatos.head())

cpf_exemplo = str(candidatos.iloc[0]['cpf'])
print(f'\nCPF escolhido: {cpf_exemplo}')

df_cpf = df_cpf_telefones[df_cpf_telefones['cpf'].astype(str) == cpf_exemplo].copy()
print(f'Linhas (com repetições) para esse CPF: {len(df_cpf)}')
print(f'Telefones únicos: {df_cpf["telefone"].nunique()}')
df_cpf

CPFs candidatos (top 5):


,cpf,n_telefones_multi
28308,12183584299170815494,3
34869,12697070039422124869,2
61172,14727769200749846417,2
7572,10580671700444919597,2
95225,17371033812469142722,2



CPF escolhido: 12183584299170815494
Linhas (com repetições) para esse CPF: 7
Telefones únicos: 3


,cpf,telefone,id_sistema,data_atualizacao
211375,12183584299170815494,531108399140214836,3094574413675758272,2023-11-17
211377,12183584299170815494,531108399140214836,13742676811738960007,2025-08-28
537460,12183584299170815494,15621800994414860797,13742676811738960007,2025-07-05
537461,12183584299170815494,15621800994414860797,1257277410380486863,2024-12-04
730272,12183584299170815494,16736468333720514268,18313131241423355789,NaT
730274,12183584299170815494,16736468333720514268,13742676811738960007,2025-07-08
730277,12183584299170815494,16736468333720514268,3094574413675758272,2023-11-17


## 3. Roda o `PhoneScorer`

A entrada é o DataFrame filtrado para o CPF — o algoritmo consolida as
repetições internamente e devolve **uma linha por telefone** com o score
decomposto em componentes interpretáveis.

In [5]:
scored = scorer.score(df_cpf)
print(f'Telefones avaliados: {len(scored)}')
scored

Telefones avaliados: 3


,cpf,telefone,rank,score_final,id_sistema,sistema_nome,score_sistema,score_atualidade,taxa_read,n_score_sistema,n_score_atualidade,n_score_read,data_atualizacao,dias_desde_atualizacao,n_sistemas,peso_sistema,peso_atualidade,peso_read
0,12183584299170815494,15621800994414860797,1,0.75,13742676811738960007,Sistema E,0.987,0.780237,1.0,0.5,0.5,1.0,2025-07-05,296,2,0.3,0.2,0.5
1,12183584299170815494,16736468333720514268,2,0.25,13742676811738960007,Sistema E,0.987,0.780237,0.0,0.5,0.5,0.0,2025-07-08,293,3,0.3,0.2,0.5
2,12183584299170815494,531108399140214836,3,0.25,13742676811738960007,Sistema E,0.987,0.780237,0.0,0.5,0.5,0.0,2025-08-28,242,2,0.3,0.2,0.5


In [6]:
top2 = scorer.top_k(df_cpf, k=2)
top2

,cpf,telefone,rank,score_final,id_sistema,sistema_nome,score_sistema,score_atualidade,taxa_read,n_score_sistema,n_score_atualidade,n_score_read,data_atualizacao,dias_desde_atualizacao,n_sistemas,peso_sistema,peso_atualidade,peso_read
0,12183584299170815494,15621800994414860797,1,0.75,13742676811738960007,Sistema E,0.987,0.780237,1.0,0.5,0.5,1.0,2025-07-05,296,2,0.3,0.2,0.5
1,12183584299170815494,16736468333720514268,2,0.25,13742676811738960007,Sistema E,0.987,0.780237,0.0,0.5,0.5,0.0,2025-07-08,293,3,0.3,0.2,0.5
